In [2]:
from itertools import product
from math import log2

In [3]:
parent_facts = {
    ("ann", "bob"),
    ("ann", "cara"),
    ("bob", "david"),
    ("cara", "erin"),
    ("david", "frank"),
    ("erin", "george"),
    ("bob", "helen"),
}

constants = sorted({x for fact in parent_facts for x in fact})

positive_examples = {
    ("ann", "david"),
    ("ann", "erin"),
    ("ann", "frank"),
    ("ann", "george"),
    ("ann", "helen"),
    ("bob", "frank"),
    ("bob", "george"),
}

all_examples = set(product(constants, repeat=2))
negative_examples = all_examples - positive_examples

In [4]:
def satisfies_literal(literal, binding):
    predicate, arg1, arg2 = literal
    if predicate != "parent":
        return []

    value1 = binding.get(arg1, arg1)
    value2 = binding.get(arg2, arg2)

    results = []
    for fact in parent_facts:
        if value1 != arg1 and value1 != fact[0]:
            continue
        if value2 != arg2 and value2 != fact[1]:
            continue

        new_binding = binding.copy()

        if arg1.isupper():
            new_binding[arg1] = fact[0]
        if arg2.isupper():
            new_binding[arg2] = fact[1]

        results.append(new_binding)

    return results

In [5]:
def covers(example, body):
    bindings = [{"X": example[0], "Y": example[1]}]

    for literal in body:
        new_bindings = []
        for binding in bindings:
            new_bindings.extend(satisfies_literal(literal, binding))
        bindings = new_bindings

        if not bindings:
            return False

    return bool(bindings)

In [6]:
def covered_examples(examples, body):
    return {example for example in examples if covers(example, body)}

In [7]:
def information_gain(p0, n0, p1, n1):
    if p1 == 0 or p1 + n1 == 0:
        return 0

    before = p0 / (p0 + n0)
    after = p1 / (p1 + n1)

    if before == 0 or after == 0:
        return 0

    return p1 * (log2(after) - log2(before))

In [8]:
def foil():
    remaining_positive = set(positive_examples)
    rules = []

    while remaining_positive:
        body = []
        covered_pos = remaining_positive.copy()
        covered_neg = negative_examples.copy()

        while covered_neg:
            variables = ["X", "Y"] + [
                variable for literal in body for variable in literal[1:]
                if variable.isupper()
            ]

            variables = list(dict.fromkeys(variables))
            new_variable = f"V{len(body)}"
            candidate_literals = set()

            for arg1 in variables + [new_variable]:
                for arg2 in variables + [new_variable]:
                    if arg1 == new_variable and arg2 == new_variable:
                        continue
                    candidate_literals.add(("parent", arg1, arg2))

            best_literal = None
            best_gain = 0

            for literal in candidate_literals:
                if literal in body:
                    continue

                candidate_body = body + [literal]
                pos = covered_examples(remaining_positive, candidate_body)
                neg = covered_examples(negative_examples, candidate_body)

                gain = information_gain(
                    len(covered_pos),
                    len(covered_neg),
                    len(pos),
                    len(neg),
                )

                if gain > best_gain:
                    best_gain = gain
                    best_literal = literal

            if best_literal is None:
                break

            body.append(best_literal)
            covered_pos = covered_examples(remaining_positive, body)
            covered_neg = covered_examples(negative_examples, body)

        if covered_pos:
            rules.append(body)
            remaining_positive -= covered_pos
        else:
            break

    return rules

In [9]:
rules = foil()

print("Learned FOIL rules:\n")
for body in rules:
    rule_body = " AND ".join(
        f"{predicate}({arg1}, {arg2})"
        for predicate, arg1, arg2 in body
    )
    print(f"grandparent(X, Y) :- {rule_body}")

Learned FOIL rules:

grandparent(X, Y) :- parent(X, V0) AND parent(V0, Y) AND parent(Y, V2)
grandparent(X, Y) :- parent(X, V0) AND parent(V0, Y)
grandparent(X, Y) :- parent(X, V0) AND parent(V0, V1) AND parent(V1, Y)
grandparent(X, Y) :- parent(X, V0) AND parent(V0, V1) AND parent(V2, X) AND parent(V3, Y) AND parent(V4, V3) AND parent(V5, V4)
